# Orders, requests, questions... What do the characters do when they speak?

In the Grimm fairy tales, does the king give **orders** and the princess make **requests**? Who asks **questions**, who makes **promises or threats**?

We classify every speech of the characters into one of six categories, with **zero-shot classification** (as in [session 7](../session07/zero_shot_variants_all.ipynb)): the model `facebook/bart-large-mnli` reads the speech (the **premise**) and scores how much it supports a sentence describing each category (the **hypotheses**), e.g. *This is an order.*

The categories and the definitions are in the [plan](plan-example-speech-acts.md).

Our strategy:
1. extract the speeches from the TEI files (`<said>`), merging the speeches interrupted by the narrator;
2. classify each speech with BART-MNLI;
3. compare female and male speakers;
4. check the model against a ground truth annotated by hand.

## 1. Find the files

In [1]:
from pathlib import Path  # Tools for files and folders.
import re  # Regular expressions.
import xml.etree.ElementTree as ET  # Reads XML files.
import pandas as pd  # Tables.

files = sorted(Path("../data/fairy-tales").glob("[0-9]*.xml"))  # The tales (the files starting with a number).
print(len(files), "tales")

22 tales


## 2. Extract the speeches

As in [speech_verbs.ipynb](speech_verbs.ipynb), we split each tale with a regular expression into `[narration, speaker, speech, narration, speaker, speech, ..., narration]`.

**Interrupted speeches.** The narrator often cuts a speech in two:

> ‘Pray lift me upon chair,’ *said he to the princess,* ‘and let me sit next to you.’

In the TEI these are two `<said>`, but it is **one** request. We merge two `<said>` when they have the same speaker and the narration between them is short (5 words or fewer, e.g. *said he to the princess,*).

In [2]:
def remove_tags(text):
    """Remove the XML tags (e.g. <persName>) and the line breaks from a text."""
    text = re.sub(r"<[^>]+>", "", text)  # Delete everything between < and >.
    return " ".join(text.split())  # Replace line breaks and multiple spaces with one space.

MAX_WORDS_BETWEEN = 5  # Narration of 5 words or fewer between two <said> of the same speaker: one speech.

def speeches_of(path):
    """Return a list of (speaker, speech) for a tale, with the interrupted speeches merged."""
    xml = path.read_text(encoding="utf-8")
    body = xml[xml.index("<body>"):]  # Skip the header.
    parts = re.split(r'<said[^>]*who="([^"]+)"[^>]*>(.*?)</said>', body, flags=re.S)
    narration = [remove_tags(p) for p in parts[0::3]]  # Positions 0, 3, 6, ...
    speakers = parts[1::3]  # Positions 1, 4, 7, ...
    speeches = [remove_tags(p) for p in parts[2::3]]  # Positions 2, 5, 8, ...

    result = []
    for i in range(len(speeches)):
        text = speeches[i].strip("‘’ ")  # Remove the quotation marks.
        same_speaker = i > 0 and speakers[i] == speakers[i - 1]
        short_narration = len(narration[i].split()) <= MAX_WORDS_BETWEEN  # The narration between speech i-1 and speech i.
        if same_speaker and short_narration:
            previous_speaker, previous_text = result[-1]
            result[-1] = (previous_speaker, previous_text + " " + text)  # Add this half to the previous speech.
        else:
            result.append((speakers[i], text))
    return result

for speaker, speech in speeches_of(files[1])[:6]:  # The first speeches of The Frog-Prince.
    print(speaker, "|", speech)

#frogprince-princess | Alas! if I could only get my ball again, I would give all my fine clothes and jewels, and everything that I have in the world.
#frogprince-frog | Princess, why do you weep so bitterly?
#frogprince-princess | Alas! what can you do for me, you nasty frog? My golden ball has fallen into the spring.
#frogprince-frog | I want not your pearls, and jewels, and fine clothes; but if you will love me, and let me live with you and eat from off your golden plate, and sleep upon your bed, I will bring you your ball again.
#frogprince-princess | What nonsense, this silly frog is talking! He can never even get out of the spring to visit me, though he may be able to get my ball for me, and therefore I will tell him he shall have what he asks.
#frogprince-princess | Well, if you will bring me my ball, I will do all you ask.


Now all the tales. We also read the gender of each character from the header, as in the other notebooks.

In [3]:
rows = []  # One row per speech.
genders = {}  # character id -> gender, for all the tales.
n_said = 0  # Number of <said> before merging, to check.

for f in files:
    root = ET.parse(f).getroot()
    for character in root.findall(".//{http://www.tei-c.org/ns/1.0}listPerson/*"):
        genders[character.get("{http://www.w3.org/XML/1998/namespace}id")] = character.get("gender")
    n_said += len(root.findall(".//{http://www.tei-c.org/ns/1.0}said"))

    for who, speech in speeches_of(f):
        rows.append({"speech_id": len(rows), "tale": f.stem, "who": who, "speech": speech})

df = pd.DataFrame(rows)
print(n_said, "<said> elements,", len(df), "speeches after merging")
df.head(10)

687 <said> elements, 561 speeches after merging


,speech_id,tale,who,speech
0,0,07_briar-rose,#briarrose-fish,"I know what your wish is, and it shall be fulf..."
1,1,07_briar-rose,#briarrose-queen,"I will have the fairies also, that they might ..."
2,2,07_briar-rose,#briarrose-thirteenth_fairy,"The king’s daughter shall, in her fifteenth ye..."
3,3,07_briar-rose,#briarrose-briar_rose,"Why, how now, good mother, what are you doing ..."
4,4,07_briar-rose,#briarrose-old_lady,"Spinning,"
5,5,07_briar-rose,#briarrose-briar_rose,How prettily that little thing turns round!
6,6,07_briar-rose,#briarrose-prince,All this shall not frighten me; I will go and ...
7,7,12_the-frog-prince,#frogprince-princess,"Alas! if I could only get my ball again, I wou..."
8,8,12_the-frog-prince,#frogprince-frog,"Princess, why do you weep so bitterly?"
9,9,12_the-frog-prince,#frogprince-princess,"Alas! what can you do for me, you nasty frog? ..."


## 3. The categories and the hypotheses

Six categories, inspired by the *speech acts* of Searle (1976). The definitions are in the [plan](plan-example-speech-acts.md); here we only write the **hypotheses**, one per category.

The **premise** is the speech itself, without the reporting clause (*said the king*, *begged she*): we want to know whether the model recognises an order from the words of the character, not from the verb chosen by the narrator.

The commented lines are earlier versions of the hypotheses, kept for comparison. On a small test (17 speeches chosen by hand, below) the short hypotheses worked best: 15 correct, against 11 and 7 for the others.

In [4]:
hypotheses = {
    "order":          "This is an order.",
    "request":        "This is a request.",
    "question":       "This is a question.",
    "statement":      "This is a statement.",
    "promise/threat": "This is a promise or a threat.",
    "exclamation":    "This is an exclamation.",
}

# Version B (11 of 17 correct):
# "order": "The speaker orders someone to do something.", "request": "The speaker begs someone to do something.",
# "question": "The speaker asks a question.", "statement": "The speaker tells a fact.",
# "promise/threat": "The speaker says what will happen in the future.", "exclamation": "The speaker cries out in surprise, joy or anger."

# Version C (7 of 17 correct):
# "order": "The speaker gives an order.", "request": "The speaker asks for a favour.",
# "question": "The speaker asks a question.", "statement": "The speaker describes something.",
# "promise/threat": "The speaker makes a promise or a threat.", "exclamation": "The speaker exclaims."

categories = list(hypotheses)  # The names of the categories.

## 4. Load the model and test it

The first run downloads the model (~1.6 GB). It runs on CPU; a GPU is faster but not required.

We test it on a few speeches whose category we decided by hand (these are **not** in the ground truth: we used them to choose the hypotheses, so they would make the model look better than it is).

In [5]:
## These first two lines are only needed in mybinder, Google Colab, or similar environments.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

def classify(speech):
    """Return the score of each category for one speech."""
    result = classifier(
        speech,                        # The premise.
        list(hypotheses.values()),     # The hypotheses are the candidate "labels".
        hypothesis_template="{}",      # They are already full sentences: nothing to add around them.
        multi_label=False,             # The categories compete: the scores sum to 1.
    )
    score_of = dict(zip(result["labels"], result["scores"]))  # hypothesis -> score
    return {category: round(score_of[h], 3) for category, h in hypotheses.items()}

tests = [
    ("order",          "Quick, Rose-red, open the door, it must be a traveller who is seeking shelter."),
    ("order",          "Put your plate nearer to me, that I may eat out of it."),
    ("order",          "Sweep me the room as clean as you can, up with the window, fling out the dead man!"),
    ("request",        "Pray lift me upon chair, and let me sit next to you."),
    ("request",        "Dear God, do help us"),
    ("request",        "Let me speak with the bridegroom."),
    ("question",       "What ails you, dear wife?"),
    ("question",       "Is Mrs Fox not at home?"),
    ("question",       "Has the gentleman red stockings on, and has a little pointed mouth?"),
    ("statement",      "I must spin this straw into gold, and I know not how."),
    ("statement",      "The better to see you with, my dear."),
    ("statement",      "No,"),
    ("promise/threat", "I have not seen it; but I will give thee a casket--open it when thy hour of need comes."),
    ("promise/threat", "If you do not do it, you shall die."),
    ("exclamation",    "What nonsense,"),
    ("exclamation",    "Ah! how good fowls are!"),
    ("exclamation",    "What a maypole!"),
]

correct = 0
for expected, speech in tests:
    scores = classify(speech)
    predicted = max(scores, key=scores.get)  # The category with the highest score.
    correct += predicted == expected
    mark = "  " if predicted == expected else "X "  # X = wrong.
    print(f"{mark}{expected:<15} {predicted:<15} {scores[predicted]:.2f}  {speech[:60]}")
print(correct, "of", len(tests), "correct")

/home/elena/Github/DHBern/text-analysis-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2378.42it/s]


  order           order           0.37  Quick, Rose-red, open the door, it must be a traveller who i
  order           order           0.30  Put your plate nearer to me, that I may eat out of it.
  order           order           0.23  Sweep me the room as clean as you can, up with the window, f
  request         request         0.45  Pray lift me upon chair, and let me sit next to you.
  request         request         0.34  Dear God, do help us
  request         request         0.33  Let me speak with the bridegroom.
  question        question        0.65  What ails you, dear wife?
  question        question        0.64  Is Mrs Fox not at home?
  question        question        0.70  Has the gentleman red stockings on, and has a little pointed
X statement       question        0.27  I must spin this straw into gold, and I know not how.
  statement       statement       0.36  The better to see you with, my dear.
  statement       statement       0.46  No,
  promise/threat  promise/thr

## 5. Classify all the speeches

This takes a few minutes on a CPU. For each speech we keep the **predicted category** and the **confidence** (score of the winner), and all the scores.

In [6]:
all_scores = []
for i, speech in enumerate(df["speech"]):
    if i % 50 == 0:
        print("Classifying speech", i, "of", len(df))
    all_scores.append(classify(speech))

scores = pd.DataFrame(all_scores, index=df.index)  # One row per speech, one column per category.
df["predicted"] = scores.idxmax(axis=1)  # The category with the highest score.
df["confidence"] = scores.max(axis=1)  # Its score.
for category in categories:
    df["score_" + category] = scores[category]

df[["who", "speech", "predicted", "confidence"]].head(10)

Classifying speech 0 of 561
Classifying speech 50 of 561
Classifying speech 100 of 561
Classifying speech 150 of 561
Classifying speech 200 of 561
Classifying speech 250 of 561
Classifying speech 300 of 561
Classifying speech 350 of 561
Classifying speech 400 of 561
Classifying speech 450 of 561
Classifying speech 500 of 561
Classifying speech 550 of 561


,who,speech,predicted,confidence
0,#briarrose-fish,"I know what your wish is, and it shall be fulf...",statement,0.316
1,#briarrose-queen,"I will have the fairies also, that they might ...",promise/threat,0.272
2,#briarrose-thirteenth_fairy,"The king’s daughter shall, in her fifteenth ye...",promise/threat,0.375
3,#briarrose-briar_rose,"Why, how now, good mother, what are you doing ...",question,0.432
4,#briarrose-old_lady,"Spinning,",exclamation,0.459
5,#briarrose-briar_rose,How prettily that little thing turns round!,exclamation,0.686
6,#briarrose-prince,All this shall not frighten me; I will go and ...,statement,0.413
7,#frogprince-princess,"Alas! if I could only get my ball again, I wou...",exclamation,0.294
8,#frogprince-frog,"Princess, why do you weep so bitterly?",question,0.616
9,#frogprince-princess,"Alas! what can you do for me, you nasty frog? ...",exclamation,0.350


In [7]:
# How many speeches per category?
df["predicted"].value_counts()

predicted
statement         164
question          115
exclamation       109
promise/threat     69
order              64
request            40
Name: count, dtype: int64

In [8]:
# Uncertain cases: the winner has less than 0.4 (the threshold is arbitrary; with 6 categories, chance is 1/6 = 0.17).
uncertain = df[df["confidence"] < 0.4]
print(len(uncertain), "uncertain speeches out of", len(df))
uncertain[["who", "speech", "predicted", "confidence"]].head(10)

312 uncertain speeches out of 561


,who,speech,predicted,confidence
0,#briarrose-fish,"I know what your wish is, and it shall be fulf...",statement,0.316
1,#briarrose-queen,"I will have the fairies also, that they might ...",promise/threat,0.272
2,#briarrose-thirteenth_fairy,"The king’s daughter shall, in her fifteenth ye...",promise/threat,0.375
7,#frogprince-princess,"Alas! if I could only get my ball again, I wou...",exclamation,0.294
9,#frogprince-princess,"Alas! what can you do for me, you nasty frog? ...",exclamation,0.350
10,#frogprince-frog,"I want not your pearls, and jewels, and fine c...",statement,0.235
11,#frogprince-princess,"What nonsense, this silly frog is talking! He ...",exclamation,0.296
12,#frogprince-princess,"Well, if you will bring me my ball, I will do ...",request,0.321
14,#frogprince-frog,"Open the door, my princess dear, Open the door...",request,0.291
15,#frogprince-princess,"There is a nasty frog, at the door, that lifte...",statement,0.277


## 6. Female and male speakers

One row per speech **and speaker** (one speech has two speakers, Hansel and Gretel together), with the gender from the header.

As in [speech_verbs.ipynb](speech_verbs.ipynb), we compare **percentages**: female characters have more speeches than male ones.

In [9]:
speaker_rows = []
for _, row in df.iterrows():
    for speaker in row["who"].split():  # Usually one speaker.
        char_id = speaker.lstrip("#")
        speaker_rows.append({"speech_id": row["speech_id"], "speaker": char_id, "gender": genders[char_id],
                             "speech": row["speech"], "predicted": row["predicted"], "confidence": row["confidence"]})
speakers_df = pd.DataFrame(speaker_rows)

# Rows = categories, columns = genders, values = % of the speeches of that gender.
table = pd.crosstab(speakers_df["predicted"], speakers_df["gender"], normalize="columns") * 100
table = table.round(1)
table.loc["(speeches)"] = speakers_df["gender"].value_counts()  # How many speeches in total, per gender.
table

gender,female,male,unspecified
predicted,,,
exclamation,22.8,14.6,27.5
order,12.0,12.2,2.5
promise/threat,13.8,9.8,17.5
question,21.7,21.5,5.0
request,7.2,7.3,5.0
statement,22.5,34.6,42.5
(speeches),276.0,246.0,40.0


In [10]:
# Who gives the most orders?
orders = speakers_df[speakers_df["predicted"] == "order"]
orders.groupby(["speaker", "gender"]).size().sort_values(ascending=False).head(10)

speaker                  gender
wolfkids-wolf            male      5
ashputtel-stepmother     female    3
hanselgretel-witch       female    3
snowwhiterosered-mother  female    3
cleverelsie-father       male      3
frogprince-frog          male      2
hanselgretel-stepmother  female    2
clevergretel-master      male      2
clevergretel-gretel      female    2
cleverelsie-mother       female    2
dtype: int64

## 7. Read the examples

Numbers are not enough: are the "orders" of the model really orders?

In [11]:
def show_examples(category, gender=None, n=5):
    """Print n speeches of a category (optionally only for one gender)."""
    part = speakers_df[speakers_df["predicted"] == category]
    if gender:
        part = part[part["gender"] == gender]
    print(len(part), "speeches classified as", category, "(" + (gender or "all genders") + ")")
    for _, row in part.head(n).iterrows():
        print(f"- {row['speaker']} ({row['confidence']:.2f}): {row['speech'][:100]}")
    print()

for gender in ["female", "male"]:
    show_examples("order", gender)
    show_examples("request", gender)

33 speeches classified as order (female)
- goosegirl-waiting-maid (0.27): Then tell one of your slaughterers to cut off the head of the horse I rode upon, for it was very unr
- goosegirl-princess (0.34): That I must not tell you or any man, or I shall lose my life.
- rapunzel-dame_gothel (0.29): Rapunzel, Rapunzel, Let down your hair to me.
- rapunzel-dame_gothel (0.29): Rapunzel, Rapunzel, Let down your hair to me.
- hanselgretel-stepmother (0.35): Get up, you sluggards! we are going into the forest to fetch wood.

20 speeches classified as request (female)
- frogprince-princess (0.32): Well, if you will bring me my ball, I will do all you ask.
- goosegirl-princess (0.36): Pray get down, and fetch me some water in my golden cup out of yonder brook, for I want to drink.
- goosegirl-princess (0.33): Pray get down, and fetch me some water to drink in my golden cup.
- goosegirl-waiting-maid (0.51): I brought her with me for the sake of her company on the road, pray give the girl some work

## 8. Ground truth

The code chooses 60 speeches at random and saves them in `ground_truth_speech_acts.csv` with an empty column `category_by_hand`, to fill in by hand (e.g. in a spreadsheet, saving again as CSV). The prediction of the model is **not** in the file, so that it does not influence you.

Read each speech **in the tale** (who speaks to whom? does the listener obey?) and write one of: `order`, `request`, `question`, `statement`, `promise/threat`, `exclamation`, following the definitions of the [plan](plan-example-speech-acts.md). Write your doubts in `notes`.

**Attention**: the file is created only if it does not exist yet, so your annotations are never overwritten.

In [12]:
gt_file = Path("ground_truth_speech_acts.csv")
if not gt_file.exists():  # Do not overwrite the annotations!
    sample = df.sample(n=60, random_state=1)  # 60 speeches at random.
    sample = sample[["speech_id", "tale", "who", "speech"]].copy()  # Without the prediction.
    sample["category_by_hand"] = ""  # Empty columns, to fill in by hand.
    sample["notes"] = ""
    sample.to_csv(gt_file, index=False)
    print("Created", gt_file, "- now annotate it!")
else:
    print(gt_file, "already exists: not overwritten.")

ground_truth_speech_acts.csv already exists: not overwritten.


## 9. Validation

- **Accuracy**: share of speeches where model and ground truth agree.
- **Confusion matrix**: rows = human, columns = model. Look at `order` ↔ `request`: the comparison of the genders depends on them.
- **Per category**: how many speeches of each human category the model got right. Accuracy can be high just because the frequent categories are right.
- **Confidence**: is the model more often right when it is confident?

In [13]:
gt = pd.read_csv(gt_file)

if gt["category_by_hand"].isna().any():  # Empty cells are read as NaN ("not a number").
    print("Annotate all the rows of", gt_file, "first.")
else:
    merged = gt.merge(df[["speech_id", "predicted", "confidence"]], on="speech_id")  # Add the prediction of the model.
    merged["correct"] = merged["predicted"] == merged["category_by_hand"]
    print("Accuracy:", merged["correct"].sum(), "of", len(merged), "=", round(merged["correct"].mean() * 100), "%")
    print()
    display(pd.crosstab(merged["category_by_hand"], merged["predicted"], rownames=["human"], colnames=["model"]))
    print("Correct per human category:")
    print(merged.groupby("category_by_hand")["correct"].agg(["sum", "count"]))
    print()
    print("Accuracy when confident (>= 0.4):", round(merged[merged["confidence"] >= 0.4]["correct"].mean() * 100), "%")
    print("Accuracy when uncertain (< 0.4):", round(merged[merged["confidence"] < 0.4]["correct"].mean() * 100), "%")
    print()
    print("Disagreements:")
    display(merged[~merged["correct"]][["speech", "category_by_hand", "predicted", "confidence", "notes"]])

Annotate all the rows of ground_truth_speech_acts.csv first.


## 10. Save the results

In [14]:
os.makedirs("results", exist_ok=True)
df.to_csv("results/speech_acts.csv", index=False)
print("Saved to results/speech_acts.csv")

Saved to results/speech_acts.csv


## Things to keep in mind
- **The model does not know who speaks to whom.** The difference between an order and a request often depends on the relation (a king and his servant, a daughter and her father), which is not in the speech.
- **One category per speech**: ‘I have not seen it; but I will give thee a casket--open it when thy hour of need comes.’ is a statement, a promise and an order at the same time.
- **Confidence ≠ correctness**: a model can be confidently wrong.
- **Prompt sensitivity**: the hypotheses were chosen on 17 speeches only. Different wordings give different results.

### Future directions
1. Add the reporting clause to the premise (e.g. *‘Open the door,’ said the king*; you can reuse the function `clauses` of [speech_verbs.ipynb](speech_verbs.ipynb)). Does the accuracy improve? Is it still the model that recognises the order, or the verb chosen by the narrator?
2. Try `multi_label=True`: each category is scored on its own, from 0 to 1. How many speeches have a high score (e.g. > 0.8) in two categories? Which pairs?
3. Rewrite the hypotheses for `order` and `request` (e.g. *The speaker expects to be obeyed.* / *The speaker asks politely.*). Check the effect on the confusion matrix, **not** on the 17 test speeches only.
4. Thoughts (`<said aloud="false">`) are classified like speeches. Which category do they get? Should they be left out?
5. Compare the **female antagonists** (stepmothers, witches) with the **kings**: who gives more orders?